# Channel Ablation — Baseline CNN

Investigates how CTC decoding degrades as fewer sEMG electrode channels are used per hand.
Every-Nth-channel selection (`ChannelSelect`) maintains uniform spatial coverage.

| Channels/hand | Val CER | Test CER | Training time |
|---|---|---|---|
| 16 (baseline) | **18.52%** | 22.28% | 3 h 51 m |
| 8 | 18.65% | 23.30% | 1 h 13 m |
| 4 | 24.88% | 27.12% | 1 h 10 m |
| 2 | 40.85% | 45.00% | 1 h 09 m |

```bash
source /home/benforbes/emg2qwerty/venv/bin/activate
cd ~/C247_mumbikaijonathanben && jupyter notebook
```

## 1. Setup & Patch

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path(os.getcwd())
while not (REPO_ROOT / 'emg2qwerty').is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PLAYGROUND = REPO_ROOT / 'Playground_Ben'
UCLA = {'blue': '#2774AE', 'gold': '#FFD100', 'dark_blue': '#003B5C', 'dark_gold': '#FFB300'}

# Patch emg2qwerty with Ben's dynamic channel count + new transforms
shutil.copy(PLAYGROUND / 'emg2qwerty/transforms.py', REPO_ROOT / 'emg2qwerty/transforms.py')
shutil.copy(PLAYGROUND / 'emg2qwerty/lightning.py',  REPO_ROOT / 'emg2qwerty/lightning.py')

# Copy ablation configs into the main config tree
for cfg in (PLAYGROUND / 'config/transforms').glob('channel_stride*.yaml'):
    shutil.copy(cfg, REPO_ROOT / 'config/transforms' / cfg.name)

print('Patched and configs copied.')
print(f'Repo root: {REPO_ROOT}')

## 2. Train All Channel Conditions

Run cells 2.1–2.4 in sequence. Each trains the full 150-epoch model for its channel count.
Skip any condition you have already trained.

### 2.1 — 16 channels/hand (baseline, no override)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'16 ch exit code: {result.returncode}')

### 2.2 — 8 channels/hand (every-other electrode)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train',
     '+transforms=channel_stride2', 'model.in_features=264', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'8 ch exit code: {result.returncode}')

### 2.3 — 4 channels/hand (every 4th electrode)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train',
     '+transforms=channel_stride4', 'model.in_features=132', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'4 ch exit code: {result.returncode}')

### 2.4 — 2 channels/hand (every 8th electrode)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train',
     '+transforms=channel_stride8', 'model.in_features=66', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'2 ch exit code: {result.returncode}')

## 3. Results from Pre-computed CSVs

In [ ]:
df = pd.read_csv(PLAYGROUND / 'results/channel_ablation_table.csv').sort_values('channels_per_hand')
print(df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Channel Ablation — Baseline TDSConvCTC', fontsize=13)

x = df['channels_per_hand'].values
axes[0].plot(x, df['val_cer_pct'],  'o-', color=UCLA['blue'],      lw=2.5, ms=8, label='Val CER')
axes[0].plot(x, df['test_cer_pct'], 's--', color=UCLA['dark_gold'], lw=2,   ms=7, label='Test CER')
for ch, v in zip(x, df['val_cer_pct'].values):
    axes[0].annotate(f'{v:.1f}%', (ch, v), textcoords='offset points', xytext=(0, 9), ha='center', fontsize=9)
axes[0].set(xlabel='Channels per hand', ylabel='CER (%)', title='CER vs Channels per Hand')
axes[0].set_xticks(x); axes[0].set_ylim(0, 55)
axes[0].legend(); axes[0].grid(True, alpha=0.25, ls='--')

axes[1].bar(df['channels_per_hand'].astype(str), df['training_time_sec'] / 3600,
            color=UCLA['blue'], edgecolor='white', width=0.5)
axes[1].set(xlabel='Channels per hand', ylabel='Training time (hrs)', title='Training Time vs Channels')
axes[1].grid(True, alpha=0.25, ls='--', axis='y')

plt.tight_layout()
plt.savefig(PLAYGROUND / 'plots/channel_ablation_final.png', dpi=150)
plt.show()